## 1. Importações

In [1]:
# Manipulação de dados
import pandas as pd
import numpy as np
import time

# Visualização (para gráficos de performance)
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Métricas de Avaliação
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    confusion_matrix
)

# Ferramentas extras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Carregar Modelo

In [2]:

caminho_dados = '../data/processed/data_model/data_model.csv'
df_model = pd.read_csv(caminho_dados, sep=';')

print(f"Dataset carregado com sucesso! Formato atual: {df_model.shape}")

Dataset carregado com sucesso! Formato atual: (208919, 83)


### 2.1. Removendo vazamento de dados por variável `classificacao_acidente`

In [3]:
# Removendo variáveis de vazamento (Data Leakage) que entregam o gabarito
colunas_vazamento = [col for col in df_model.columns if 'classificacao_acidente' in col or 'tipo_envolvido' in col or 'estado_fisico' in col]

df_model = df_model.drop(columns=colunas_vazamento, errors='ignore')

# Agora sim, defina X e y
y = df_model['houve_obito']
X = df_model.drop(columns=['houve_obito'])

In [4]:
# Isolando a variável alvo e as features explicativas
y = df_model['houve_obito']
X = df_model.drop(columns=['houve_obito'])

print(f"Número de features disponíveis para o modelo: {X.shape[1]}")
print(f"Quantidade total de registros: {X.shape[0]}")

Número de features disponíveis para o modelo: 74
Quantidade total de registros: 208919


In [5]:

# Divisão dos dados mantendo a proporção de classes do alvo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Registros reservados para Treinamento: {X_train.shape[0]}")
print(f"Registros reservados para Teste (Validação): {X_test.shape[0]}")

Registros reservados para Treinamento: 167135
Registros reservados para Teste (Validação): 41784


In [6]:

# Instanciando o escalonador e ajustando os limites com base no treino
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Padronização das features concluída com sucesso.")

Padronização das features concluída com sucesso.


In [7]:

# Configuração dos três algoritmos avançados com balanceamento de peso
modelos_avancados = {
    "Random Forest": RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=33, n_jobs=-1, random_state=42, eval_metric='auc'),
    "LightGBM": LGBMClassifier(class_weight='balanced', n_jobs=-1, random_state=42, verbose=-1)
}

# Lista para consolidar o histórico de performance
resultados_modelos = []

print("Iniciando o ciclo de treinamento e validação...\n")

for nome, modelo in modelos_avancados.items():
    tempo_inicio = time.time()
    
    # Ajuste do modelo
    print(f"Treinando o algoritmo: {nome}...")
    modelo.fit(X_train_scaled, y_train)
    
    # Predições de classe e probabilidades
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    
    tempo_fim = time.time()
    duracao = tempo_fim - tempo_inicio
    
    # Coleta de métricas específicas
    resultados_modelos.append({
        "Modelo": nome,
        "Acurácia": round(accuracy_score(y_test, y_pred), 4),
        "Precisão (Óbito)": round(precision_score(y_test, y_pred), 4),
        "Recall (Óbito)": round(recall_score(y_test, y_pred), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4),
        "Tempo de Treino (s)": round(duracao, 2)
    })

print("\nTodos os modelos avançados foram treinados e avaliados!")

Iniciando o ciclo de treinamento e validação...

Treinando o algoritmo: Random Forest...
Treinando o algoritmo: XGBoost...
Treinando o algoritmo: LightGBM...

Todos os modelos avançados foram treinados e avaliados!


d:\Projetos\PredicaoAcidentesPRF\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Projetos\PredicaoAcidentesPRF\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
# Convertendo os resultados coletados para um DataFrame
df_performance = pd.DataFrame(resultados_modelos)

# Adicionando manualmente os dados reais da sua Regressão Logística (Baseline corrigido)
dados_baseline = {
    "Modelo": "Regressão Logística (Baseline)",
    "Acurácia": 0.7600,
    "Precisão (Óbito)": 0.0900,
    "Recall (Óbito)": 0.7800,
    "AUC-ROC": 0.8552,
    "Tempo de Treino (s)": 1.20 # Tempo médio estimado
}

# Inserindo o baseline na primeira linha para fins de comparação direta
df_performance = pd.concat([pd.DataFrame([dados_baseline]), df_performance], ignore_index=True)

# Ordenando pelo critério do AUC-ROC
df_performance = df_performance.sort_values(by="AUC-ROC", ascending=False).reset_index(drop=True)

print("--- PAINEL COMPARATIVO FINAL DOS MODELOS ---")
display(df_performance)

--- PAINEL COMPARATIVO FINAL DOS MODELOS ---


,Modelo,Acurácia,Precisão (Óbito),Recall (Óbito),AUC-ROC,Tempo de Treino (s)
0,LightGBM,0.8092,0.1095,0.7697,0.8754,0.77
1,XGBoost,0.8560,0.1293,0.6794,0.8633,0.70
2,Random Forest,0.9711,0.5618,0.0814,0.8598,5.34
3,Regressão Logística (Baseline),0.7600,0.0900,0.7800,0.8552,1.20
